In [ ]:
# thread: program에서 일을 처리하는 단위(많을수록 동시에 일 처리 가능)
# GIL: 한 번에 하나의 thread만 python code를 실행하게 만듦
# python은 GIL로 인해 thread가 GIL을 얻지 못하면 일을 수행할 수 없음
# 비동기 방식을 쓰면 한 작업의 대기 시간 발생 시 다음 작업이 바로 이어받을 수 있다.
# 대기 중인 작업은 OS가 처리하고, thread는 다른 작업을 실행한다.

# 비동기 함수는 I/O 작업(CPU가 아니라 외부(네트워크, 디스크, DB)를 기다리는 작업)이 있을 때 작성한다.
# main함수부터 async로 작성하고 module로 분리해서 app에서 import 한다.

In [3]:
# 비동기 함수
async def greet():
    return "hello"

# async만 있으면 coroutine 객체만 반환하고 실행은 안 됨
result = greet()
print(type(result))  # <class 'coroutine'> - hello가 출력되지 않음
# RuntimeWarning: coroutine 'greet' was never awaited

<class 'coroutine'>


In [ ]:
import asyncio

# await 
# 1. coroutine 실행
# 2. event loop가 해당 coroutine이 끝날 때까지 기다렸다가 결과를 반환
# ※ 이때 계속 thread를 잡고 있는 것이 아니라 eventloop를 거쳐 os에 맡김
async def fetch_data():
  print("요청 시작")
  await asyncio.sleep(2) # 여기서 양보
  print("응답 수신")
  return "data"
  
async def main():
  result = await fetch_data()
  print(result)

# 일반 python script로 실행하면 동작 잘 됨
# asyncio.run(main())
await main()

요청 시작
응답 수신
data


In [ ]:
# 동기: DB 기다리는 동안 thread 잠김 -> 10명 요청 시 한 명씩 줄줄이 밀림
# 비동기: DB 기다리는 동안 thread 양보 -> 10명 요청 거의 동시 처리 가능
# 즉, await는 "기다리는 동안 다른 요청도 처리해" 라는 의미

# async/await 없으면 (동기 방식)
# @app.get("/data")
# def get_data():  # 일반 함수
#     result = fetch_from_db()  # DB 응답 기다리는 동안 thread 완전히 잠김
#     return result

# async/await 있으면
# @app.get("/data")
# async def get_data():
#     result = await fetch_from_db()  # 기다리는 동안 양보!
#     return result

In [ ]:
# background
import asyncio

async def fetch_data(url):
  await asyncio.sleep(1)
  return f"{url} done"

async def main():
  # create_task로 background에 등록
  # 중간에 await가 없으면 event_loop가 비어지지 않으므로 계속해서 background에서 대기(main이 event_loop를 쥐고 있으므로)
  task = asyncio.create_task(fetch_data("api/1"))
  # task를 await로 실행하면 event_loop가 main을 놓기 때문에 event_loop가 create_task 안의 coroutine 객체를 background 실행으로 위임
  # 그 전에 await가 있으면 background를 실행시킬 수 있으므로 실행된 결과가 반환
  result = await task
  return result

await main()

'api/1 done'

In [2]:
# 동시 처리
import asyncio
async def task_a():
  print("A: start")
  await asyncio.sleep(1)
  print("A: end")

async def task_b():
  print("B:start")
  await asyncio.sleep(2)
  print("B:end")

async def main():
  # 동시 실행
  await asyncio.gather(task_a(), task_b())

await main()

A: start
B:start
A: end
B:end


In [ ]:
# async with: 열고 닫는 작업이 비동기일 때 사용
# with는 context manager로써 쓰고 나서 자동으로 정리해준다.
# async for: 다음 데이터가 올 때까지 기다리면서 반복 (스트리밍)

In [ ]:
import asyncio
import time
import httpx

URLS=[
  "https://www.google.com",
  "https://www.github.com",
  "https://www.naver.com",
]

async def fetch_url(url):
  # AsyncClient의 모든 method는 비동기
  # async with 진입 시 내부적으로 await __aenter__() 호출하여 연결 풀 열기
  async with httpx.AsyncClient() as client:
    response = await client.get(url)
    return {"url": url, "status": response.status_code}

async def fetch_all(urls):
  # fetch_url()은 async def이므로 호출 시 coroutine 반환 → coroutine list 생성
  tasks = [fetch_url(url) for url in urls]
  # *: unpacking 연산자(js의 spread 연산자와 동일)
  # gather는 tuple을 반환
  result = await asyncio.gather(*tasks)
  return list(result)

async def main():
  start = time.time()
  results = await fetch_all(URLS)
  print(time.time() - start, results)

await main()

0.6624410152435303 [{'url': 'https://www.google.com', 'status': 200}, {'url': 'https://www.github.com', 'status': 301}, {'url': 'https://www.naver.com', 'status': 200}]


In [ ]:
import asyncio

async def check_server(name: str, should_fail: bool) -> str:
    await asyncio.sleep(1)
    if should_fail:
        raise ConnectionError(f"{name} 서버 연결 실패")

    else:
        return f"{name} 서버 정상 (200 OK)"
async def check_all_servers(servers: list[tuple]) -> list:
    tasks = [check_server(*server) for server in servers]
    # 예외가 발생해도 모든 작업을 실패로 돌리지 않고 예외 발생한 것을 추출하여 상황을 알려준다.
    return await asyncio.gather(*tasks, return_exceptions=True)

# main 함수를 완성하세요
async def main():
    servers = [
        ("서버A", False),
        ("서버B", True),
        ("서버C", False),
        ("서버D", True),
        ("서버E", False),
    ]
    
    results = await check_all_servers(servers)
    
    success = 0
    fail = 0
    for r in results:
        if isinstance(r, Exception):
            print(f"[실패] {r}")
            fail += 1
        else:
            print(f"[성공] {r}")
            success += 1

    print(f"\n결과: 성공 {success}건, 실패 {fail}건")

await main()

[성공] 서버A 서버 정상 (200 OK)
[실패] 서버B 서버 연결 실패
[성공] 서버C 서버 정상 (200 OK)
[실패] 서버D 서버 연결 실패
[성공] 서버E 서버 정상 (200 OK)

결과: 성공 3건, 실패 2건
